# 03 — Outlier policy & residential filter

Define a default filter that isolates *residential-scale* sales from portfolio / bulk / symbolic-transfer noise.

**Proposed baseline filter:**
- `n_rows <= 10` — drop portfolio / bulk deals
- `valeur_fonciere >= 1000` — drop symbolic transfers (gifts, divisions, €1 transfers)
- `nature_mutation = 'Vente'` — plain sales only (keeps 93% of mutations)
- `n_communes = 1` — a single sale should touch one commune

In [ ]:
import duckdb, pandas as pd
from pathlib import Path
PARQUET_GLOB = str(Path.cwd().parent / "data" / "parquet" / "year=*" / "*.parquet")
con = duckdb.connect()
con.execute(f"CREATE VIEW dvf AS SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)")
print("Row count:", con.execute("SELECT COUNT(*) FROM dvf").fetchone()[0])


In [ ]:
# English translations of DVF categorical values
VALUE_TRANSLATIONS = {
    "Vente": "Sale",
    "Vente en l'état futur d'achèvement": "Off-plan (VEFA, pre-construction)",
    "Echange": "Exchange / swap",
    "Vente terrain à bâtir": "Sale of building land",
    "Adjudication": "Auction",
    "Expropriation": "Expropriation",
    "Maison": "House",
    "Appartement": "Apartment",
    "Dépendance": "Outbuilding / annex",
    "Local industriel. commercial ou assimilé": "Commercial / industrial",
}
def tr(col):
    return col.map(VALUE_TRANSLATIONS).fillna("")


## 1. Categorical rundown (columns with ≤10 distinct values)
What flavors of transaction are in here?

In [ ]:
for col in ['nature_mutation', 'type_local', 'year']:
    df = con.execute(f'''
      SELECT COALESCE(CAST("{col}" AS VARCHAR), '(null)') AS value,
             COUNT(*) AS rows,
             ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM dvf), 3) AS pct
      FROM dvf GROUP BY "{col}" ORDER BY rows DESC
    ''').df()
    df['english'] = tr(df['value'])
    print(f'\n=== {col} ===')
    print(df.to_string(index=False))

(`code_type_local` mirrors `type_local` 1:1, and `ancien_*` fields are >99.99% null — records of rare post-merge commune renames.)

## 2. Build a sale-level view with filter flags

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE sales AS
SELECT
  id_mutation,
  ANY_VALUE(date_mutation)     AS date,
  ANY_VALUE(year)              AS year,
  ANY_VALUE(valeur_fonciere)   AS valeur,
  ANY_VALUE(nature_mutation)   AS nature,
  ANY_VALUE(code_departement)  AS dept,
  ANY_VALUE(nom_commune)       AS commune,
  COUNT(*)                     AS n_rows,
  COUNT(DISTINCT code_commune) AS n_communes
FROM dvf GROUP BY id_mutation
''')
con.execute('SELECT COUNT(*) FROM sales').fetchone()

## 3. Impact of each filter

In [ ]:
con.execute('''
SELECT
  COUNT(*)                                             AS all_sales,
  SUM(n_rows > 10)::BIGINT                             AS drop_n_rows_gt_10,
  SUM(valeur < 1000 OR valeur IS NULL)::BIGINT         AS drop_valeur_lt_1000,
  SUM(n_communes > 1)::BIGINT                          AS drop_multi_commune,
  SUM(nature != 'Vente')::BIGINT                       AS drop_not_vente,
  SUM(n_rows<=10 AND valeur>=1000
      AND n_communes=1 AND nature='Vente')::BIGINT     AS keep_residential_strict,
  SUM(n_rows<=10 AND valeur>=1000)::BIGINT             AS keep_loose_filter
FROM sales
''').df()

## 4. Sanity: what's at the n_rows = 11..15 border?
If the filter is correct, borderline sales should be mostly non-residential (portfolio + outbuilding clusters, not family homes).

In [ ]:
con.execute('''
WITH borderline AS (SELECT id_mutation FROM sales WHERE n_rows BETWEEN 11 AND 15)
SELECT COALESCE(type_local, '(terrain/parcel)') AS type_local,
       COUNT(*) AS rows
FROM dvf JOIN borderline USING (id_mutation)
GROUP BY type_local ORDER BY rows DESC
''').df()

## 5. Known data quality issue: €14.15B "sale"
A single mutation recorded at €14.15 **billion** in Marseille 8e survives our filter (it's only 3 rows). Almost certainly a misplaced decimal at data entry time. For residential analyses we should either cap valeur or flag this specific `id_mutation`.

In [ ]:
con.execute('''
SELECT id_mutation, date, ROUND(valeur) AS valeur, nature, n_rows, commune, dept
FROM sales
WHERE n_rows <= 10 AND valeur >= 1000
ORDER BY valeur DESC LIMIT 10
''').df()

## 6. Test query, before vs. after filter
2-Maison, ≥5 pieces, Vente. Filter trims <3% of rows and the median barely moves — the filter cleans noise without warping the signal.

In [ ]:
con.execute('''
WITH q AS (
  SELECT id_mutation, MIN(nombre_pieces_principales) AS min_pieces, COUNT(*) AS n_mais
  FROM dvf WHERE type_local = 'Maison' GROUP BY id_mutation
)
SELECT s.year,
  SUM(1)::BIGINT AS before_filter,
  SUM(s.n_rows <= 10 AND s.valeur >= 1000)::BIGINT AS after_filter,
  ROUND(AVG(s.valeur)) AS avg_before,
  ROUND(AVG(CASE WHEN s.n_rows<=10 AND s.valeur>=1000 THEN s.valeur END)) AS avg_after,
  ROUND(MEDIAN(s.valeur)) AS med_before,
  ROUND(MEDIAN(CASE WHEN s.n_rows<=10 AND s.valeur>=1000 THEN s.valeur END)) AS med_after
FROM sales s JOIN q USING (id_mutation)
WHERE q.n_mais = 2 AND q.min_pieces >= 5 AND s.nature = 'Vente'
GROUP BY s.year ORDER BY s.year
''').df()

## 7. Bonus — price per m² for single-Maison sales (default residential filter)
Use this as a candidate metric for the webapp. Only Maison sales where the mutation contains exactly one Maison row, built surface reported.

In [ ]:
con.execute('''
WITH maison_only AS (
  SELECT id_mutation,
         ANY_VALUE(surface_reelle_bati)     AS surface,
         ANY_VALUE(nombre_pieces_principales) AS pieces,
         COUNT(*) AS n_mais
  FROM dvf WHERE type_local = 'Maison' GROUP BY id_mutation
)
SELECT s.year,
  COUNT(*) AS n_sales,
  ROUND(MEDIAN(s.valeur / m.surface)) AS median_eur_per_m2,
  ROUND(AVG(s.valeur / m.surface))    AS avg_eur_per_m2
FROM sales s JOIN maison_only m USING (id_mutation)
WHERE m.n_mais = 1 AND m.surface > 10
  AND s.nature = 'Vente' AND s.valeur BETWEEN 1000 AND 20000000
  AND s.n_rows <= 10
GROUP BY s.year ORDER BY s.year
''').df()